In [19]:
import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

In [20]:
# Activate Conda Env "Ultra" As the Kernel
import torch
import pandas as pd
from explainers.data_util import batched_k_hop_subgraph
import networkx as nx
from pyvis.network import Network

In [21]:
dataset = torch.load('/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/wn18rr_dataset.pt')
id2entity, id2relation = torch.load('/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/wn18rr_id2name.pt')

In [22]:
split = 'valid'
ratio = 50

if split == 'valid':
    data_index = 1
if split == 'test':
    data_index = 2

# root_dir = "/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/explanation/EdgeRAWExplainer/NBFNet/FB15k-237/2025-10-06-14-48-53-edge-rerun-5E/"
# root_dir = "/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/explanation/ola/finetune/"

root_dir = "/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/docs/RAWExplainer/NBFNet/WN18RR/RAWExplainer-Hyp-Jub1_5/"
output_file = f"{root_dir}/{split}_output_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
explanation_file = f"{root_dir}/{split}_explanations_factual_eval_hard_edge_mask_top_k_{ratio}.pt"

explanations = torch.load(explanation_file, map_location=torch.device('cpu'))
outputs = torch.load(output_file, map_location=torch.device('cpu'))
output_df = pd.DataFrame(outputs)

data = dataset[data_index]

In [23]:
def get_row_h_r_t(index, output_file_df):
    row = output_file_df.iloc[index]
    if row['Mode'] == 1:
        rel = id2relation[row['Rel']]
    else:
        rel = id2relation[row['Rel']]+'_inv'
    head_id = row['Heads']
    tail_id = row['Tails']
    head = id2entity[head_id]
    tail = id2entity[tail_id]
    return row, head, rel, tail, head_id, tail_id

In [24]:
from nbfnet.tasks import edge_match

def vizualize_explanation_fb(
    index,
    data,
    id2entity,
    id2relation,
    num_relations,
    output_df,
    expl,
):
    # The GNN_eval Result
    row = output_df.iloc[index]
    if row["Mode"] == 1:
        rel = id2relation[row["Rel"]]
    else:
        rel = id2relation[row["Rel"]] + "_inv"
    head_id = row["Heads"]
    tail_id = row["Tails"]
    rel_id = row["Rel"]
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    # Print the ranking
    print(f"*** Query: {head, rel}. Answer: {tail} ***")

    nodes, edges = batched_k_hop_subgraph(
        torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes
    )

    print(
        f"Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}"
    )
    full_edge_index = data.edge_index[:, edges.squeeze()]

    if row["Mode"] == 1:
        query = torch.tensor([[head_id, rel_id]])
    else:
        query = torch.tensor([[head_id, rel_id + num_relations // 2]])
    # find the matching explanations
    all_query = expl[0]
    row_id, num_match = edge_match(all_query.T, query.T)
    assert torch.all(num_match == 1)
    expl = expl[1][row_id]

    # expl = explanations[index]
    # query = expl[:2]
    # expl = expl[2:]
    # check if query matches the output.
    # Remove any non-edges:
    non_edge_mask = expl < 0
    expl = expl[~non_edge_mask]
    # Check that the explanation are restricted only to within the ego_network (not necessary)
    # assert torch.all(torch.logical_and(expl, ~edges.squeeze()) == False)
    expl_edge_index = data.edge_index[:, expl]
    expl_edge_type = data.edge_type[expl]

    print(
        f"Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}"
    )
    print(
        f"Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}"
    )

    # Create a directed graph
    G = nx.MultiDiGraph()

    # Add edges from expl_edge_index
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        relation = expl_edge_type[i].item()
        if relation < num_relations // 2:
            rel_name = id2relation[relation]
        else:
            rel_name = id2relation[relation - num_relations // 2] + "_inv"
        # G.add_edge(source, target,arrows='to')
        G.add_edge(source, target, title=rel_name)

    node_ids = torch.unique(expl_edge_index)
    attrs = {}

    nx.set_node_attributes(G, attrs, name="title")
    attrs = {head_id: "#FF0000", tail_id: "#00FF00"}
    nx.set_node_attributes(G, attrs, name="color")
    attrs = {head_id: head, tail_id: tail}
    nx.set_node_attributes(G, attrs, name="title")

    try:
        paths = list(nx.all_simple_paths(G, source=head_id, target=tail_id, cutoff=6))

        attrs = {}
        if len(paths) > 0:
            connected = True
        else:
            connected = False

        print(f"Is the tail connected to the head? {connected}")
        for path in paths:
            src = head_id
            for node in path[1:]:
                edge = (src, node)
                attrs[edge] = {"color": "#b27ebd"}
                src = node

        nx.set_edge_attributes(G, attrs)
    except:
        print(f"Is the tail connected to the head? {False}")

    return G

In [8]:
# # Preprocess the explanations
# from notebooks.vizualize import prepare_expl
# expl = prepare_expl(explanations)

# index=1073
# G = vizualize_explanation_fb(index, data,
#     id2entity,
#     id2relation,
#     dataset.num_relations,
#     output_df,
#     expl,)

In [14]:
from nbfnet.tasks import edge_match
from igraph import Graph
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection

def vizualize_explanation_viz(
    index,
    data,
    id2entity,
    id2relation,
    num_relations,
    output_df,
    expl,
):
    # The GNN_eval Result
    row = output_df.iloc[index]
    if row["Mode"] == 1:
        rel = id2relation[row["Rel"]]
    else:
        rel = id2relation[row["Rel"]] + "_inv"
    head_id = row["Heads"]
    tail_id = row["Tails"]
    rel_id = row["Rel"]
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    # Print the ranking
    print(f"*** Query: {head, rel}. Answer: {tail} ***")

    nodes, edges = batched_k_hop_subgraph(
        torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes
    )

    print(
        f"Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}"
    )
    full_edge_index = data.edge_index[:, edges.squeeze()]

    if row["Mode"] == 1:
        query = torch.tensor([[head_id, rel_id]])
    else:
        query = torch.tensor([[head_id, rel_id + num_relations // 2]])
    # find the matching explanations
    all_query = expl[0]
    row_id, num_match = edge_match(all_query.T, query.T)
    assert torch.all(num_match == 1)
    expl = expl[1][row_id]

    # expl = explanations[index]
    # query = expl[:2]
    # expl = expl[2:]
    # check if query matches the output.
    # Remove any non-edges:
    non_edge_mask = expl < 0
    expl = expl[~non_edge_mask]
    # Check that the explanation are restricted only to within the ego_network (not necessary)
    # assert torch.all(torch.logical_and(expl, ~edges.squeeze()) == False)
    expl_edge_index = data.edge_index[:, expl]
    expl_edge_type = data.edge_type[expl]

    # print(
    #     f"Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}"
    # )
    # print(
    #     f"Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}"
    # )

    # Create a directed graph
    G = nx.MultiDiGraph()

    # Add edges from expl_edge_index
    print("Len of edges expl: ", expl_edge_index.shape[1])
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        relation = expl_edge_type[i].item()
        if relation < num_relations // 2:
            rel_name = id2relation[relation]
        else:
            rel_name = id2relation[relation - num_relations // 2] + "_inv"
        # G.add_edge(source, target,arrows='to')
        G.add_edge(source, target, title=rel_name)


    head_id_int = int(head_id)
    tail_id_int = int(tail_id)

    # Get explanation edges and nodes
    expl_edges = {(int(expl_edge_index[0, i].item()), int(expl_edge_index[1, i].item())) 
                  for i in range(expl_edge_index.shape[1])}

    # Get nodes that are part of explanation
    expl_nodes = set()
    for src, tgt in expl_edges:
        expl_nodes.add(src)
        expl_nodes.add(tgt)
    
    # Always include head and tail nodes
    expl_nodes.add(head_id)
    expl_nodes.add(tail_id)

    # Build igraph ONLY with explanation subgraph for layout
    expl_edge_list = list(expl_edges)
    
    # Also build full graph for background
    full_edge_list = [(int(full_edge_index[0, i].item()), int(full_edge_index[1, i].item())) 
                      for i in range(full_edge_index.shape[1])]

    print(len(full_edge_list))
    print(len(expl_edge_list))

    # concate full_edge_list and expl_edge_list
    full_edge_list = full_edge_list + expl_edge_list

    # Use full graph for layout computation (to get consistent positions)
    g_full = Graph.TupleList(full_edge_list, directed=True)
    np.random.seed(9999)
    layout = g_full.layout_fruchterman_reingold()
    
    # Convert layout to numpy arrays
    coords = np.array(layout.coords)
    
    # Map node names to indices for full graph
    name_to_idx = {int(g_full.vs[i]['name']): i for i in range(len(g_full.vs))}
    
    # Get all node IDs
    all_node_ids = set(name_to_idx.keys())
    
    fig, ax = plt.subplots(figsize=(15, 10), facecolor='white', dpi=150)
    ax.set_facecolor('white')
    
    # Draw background edges (all edges not in explanation)
    bg_segs = []
    for src, tgt in full_edge_list:
        # if (src, tgt) not in expl_edges:
        seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
        bg_segs.append(seg)
    
    if bg_segs:
        lc_bg = LineCollection(bg_segs, colors='#DDDDDD', alpha=0.15, linewidths=0.3, zorder=1)
        ax.add_collection(lc_bg)
    
    # Draw explanation edges
    expl_segs = []
    print(expl_edge_list)
    for src, tgt in expl_edge_list:
        seg = [coords[name_to_idx[src]], coords[name_to_idx[tgt]]]
        expl_segs.append(seg)
    
    if expl_segs:
        lc_expl = LineCollection(expl_segs, colors='#0066FF', alpha=0.8, linewidths=1.5, zorder=3)
        ax.add_collection(lc_expl)
    
    # Draw background nodes (not in explanation)
    bg_node_ids = all_node_ids - expl_nodes
    if bg_node_ids:
        bg_coords = np.array([coords[name_to_idx[nid]] for nid in bg_node_ids])
        ax.scatter(bg_coords[:, 0], bg_coords[:, 1], 
                  c='#CCCCCC', s=15, zorder=2, alpha=0.3, edgecolors='none')
    
    # Draw explanation nodes (light blue)
    expl_only_nodes = expl_nodes - {head_id_int, tail_id_int}
    if expl_only_nodes:
        expl_coords = np.array([coords[name_to_idx[nid]] for nid in expl_only_nodes])
        ax.scatter(expl_coords[:, 0], expl_coords[:, 1], 
                  c='#87CEEB', s=100, zorder=4, edgecolors='white', linewidths=1)
    
    # Draw head (red)
    if head_id_int in name_to_idx:
        hx, hy = coords[name_to_idx[head_id_int]]
        ax.scatter(hx, hy, c='#FF0000', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    # Draw tail (green)
    if tail_id_int in name_to_idx:
        tx, ty = coords[name_to_idx[tail_id_int]]
        ax.scatter(tx, ty, c='#00FF00', s=200, zorder=5, edgecolors='white', linewidths=2)
    
    # Add legend
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#FF0000', 
               markersize=12, label='Head (Query)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#00FF00', 
               markersize=12, label='Tail (Answer)', markeredgecolor='white', markeredgewidth=1.5),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#87CEEB', 
               markersize=8, label='Explanation Nodes', markeredgecolor='white', markeredgewidth=1),
        Line2D([0], [0], color='#0066FF', linewidth=2, label='Explanation Edges'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#CCCCCC', 
               markersize=5, label='Background Nodes', alpha=0.5),
        Line2D([0], [0], color='#DDDDDD', linewidth=1, label='Background Edges', alpha=0.3),
    ]
    ax.legend(handles=legend_elements, loc='upper right', framealpha=0.9, fontsize=10)
    
    ax.axis('off')
    ax.autoscale()
    plt.tight_layout()
    plt.savefig(f'./plots/new/graph_wn_raw_{index}_full_new.png', facecolor='white', bbox_inches='tight')
    plt.close()
    
    return row['Ranking']

In [15]:
# Preprocess the explanations
from notebooks.vizualize import prepare_expl
expl = prepare_expl(explanations)

index=56
G = vizualize_explanation_viz(index, data,
    id2entity,
    id2relation,
    dataset.num_relations,
    output_df,
    expl,)

*** Query: ('satyridae.n.01', '_hypernym'). Answer: arthropod_family.n.01 ***
Is the tail included in the 6-hop neighbor of head in the original graph? True
Len of edges expl:  100
19986
100
[(35273, 916), (34857, 4190), (2373, 3217), (30519, 916), (20741, 26951), (16233, 20890), (20741, 1842), (25331, 916), (14384, 23043), (23043, 14384), (1943, 3217), (916, 40699), (33649, 14384), (14384, 2373), (1842, 20741), (902, 20741), (1976, 916), (30750, 14384), (34857, 14384), (4190, 34857), (916, 7251), (558, 20741), (916, 1976), (7251, 916), (916, 26303), (26951, 20741), (916, 700), (3217, 1943), (14384, 30750), (3217, 2373), (700, 20741), (20741, 1976), (916, 31217), (20741, 902), (1943, 4190), (916, 21219), (16233, 14384), (24782, 20741), (20741, 26303), (14384, 16233), (14384, 35127), (28129, 14384), (24847, 14384), (916, 558), (916, 25331), (1842, 916), (902, 916), (20741, 31217), (14633, 20741), (16233, 916), (14384, 34857), (25331, 20741), (16521, 916), (26951, 916), (3217, 14384), (2

In [25]:
from nbfnet.tasks import edge_match

def vizualize_explanation_count(
    index,
    data,
    id2entity,
    id2relation,
    num_relations,
    output_df,
    expl,
):
    # The GNN_eval Result
    row = output_df.iloc[index]
    if row["Mode"] == 1:
        rel = id2relation[row["Rel"]]
    else:
        rel = id2relation[row["Rel"]] + "_inv"
    head_id = row["Heads"]
    tail_id = row["Tails"]
    rel_id = row["Rel"]
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    # Print the ranking
    print(f"*** Query: {head, rel}. Answer: {tail} ***")

    nodes, edges = batched_k_hop_subgraph(
        torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes
    )

    print(
        f"Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}"
    )
    full_edge_index = data.edge_index[:, edges.squeeze()]

    if row["Mode"] == 1:
        query = torch.tensor([[head_id, rel_id]])
    else:
        query = torch.tensor([[head_id, rel_id + num_relations // 2]])
    # find the matching explanations
    all_query = expl[0]
    row_id, num_match = edge_match(all_query.T, query.T)
    assert torch.all(num_match == 1)
    expl = expl[1][row_id]

    # expl = explanations[index]
    # query = expl[:2]
    # expl = expl[2:]
    # check if query matches the output.
    # Remove any non-edges:
    non_edge_mask = expl < 0
    expl = expl[~non_edge_mask]
    # Check that the explanation are restricted only to within the ego_network (not necessary)
    # assert torch.all(torch.logical_and(expl, ~edges.squeeze()) == False)
    expl_edge_index = data.edge_index[:, expl]
    expl_edge_type = data.edge_type[expl]

    print(
        f"Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}"
    )
    print(
        f"Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}"
    )

    # Create a directed graph
    G = nx.MultiDiGraph()

    # Add edges from expl_edge_index
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        relation = expl_edge_type[i].item()
        if relation < num_relations // 2:
            rel_name = id2relation[relation]
        else:
            rel_name = id2relation[relation - num_relations // 2] + "_inv"
        # G.add_edge(source, target,arrows='to')
        G.add_edge(source, target, title=rel_name)

    node_ids = torch.unique(expl_edge_index)
    
    for i in range(expl_edge_index.shape[1]):
        src = int(expl_edge_index[0, i].item())
        dst = int(expl_edge_index[1, i].item())
        G.add_edge(src, dst)
    
    comps = list(nx.weakly_connected_components(G))
    comp_sizes = sorted([len(c) for c in comps], reverse=True)
    
    results = {
        'num_components': len(comps),
        'component_sizes_desc': comp_sizes,
        'contains_head': any(head_id in c for c in comps),
        'contains_tail': any(tail_id in c for c in comps)
    }

    return results

In [26]:
# Preprocess the explanations
from notebooks.vizualize import prepare_expl
expl = prepare_expl(explanations)

index=65
r=vizualize_explanation_count(index, data,
    id2entity,
    id2relation,
    dataset.num_relations,
    output_df,
    expl,)

print(r)

*** Query: ('trademark.n.02', '_member_of_domain_usage'). Answer: rolaids.n.01 ***
Is the tail included in the 6-hop neighbor of head in the original graph? True
Is the head included in the Explanation? True
Is the tail included in the Explanation? True
{'num_components': 1, 'component_sizes_desc': [20], 'contains_head': True, 'contains_tail': True}
